# B6 — Virtual saturation mutagenesis of every human SH3 domain (roadmap L3/L4, G4)

**Question.** What would a deep mutational scan of every SH3 domain in the atlas look like, and
where do the cancer mutations that actually occur sit within that landscape?

**What it does.** Predicts ΔΔG_fold, with its conformal interval, for **all 19 substitutions at
every position** of the 310 atlas SH3 domains (about 350,000 variants) with the folding head from
B1. The features are exactly those used for the observed variants (embeddings and ESM log-odds are
per domain, so nothing new has to be run through ESM), and the notebook **checks** that the
observed atlas variants get the same prediction as in B1.

It then maps the family-wide landscape on the 56-position GRB2-SH3 frame, and compares observed
cancer variants with all possible substitutions. Because tumours can mostly reach only amino-acid
changes that need one nucleotide substitution, the comparison is also made against the
**single-nucleotide-reachable** subset (from the genetic code; the actual codons are not known).
This is **not** a mutational-opportunity model (roadmap G1 needs trinucleotide context).

**Inputs:** A2 embeddings/log-odds/features, B1 models and atlas predictions, B3 conformal quantile.
**Runtime:** about 5 min (CPU is fine). **Outputs:** `sh3_virtual_dms.parquet`, `family_position_map.csv`,
`domain_summary.csv`, `RESULTS_B6.md`. Explanation: `docs/B6_virtual_saturation_mutagenesis.md`.

In [1]:
# ── setup (identical in every notebook) ─────────────────────────────────────────
ROOT   = '/content/drive/MyDrive/sh3_repro'   # the Drive folder from UPLOAD_GUIDE.md
RESUME = True                                  # reuse finished steps after a disconnect

import os, sys, json, time, importlib
from google.colab import drive; drive.mount('/content/drive')
os.system('pip -q install biopython')
sys.path.insert(0, f'{ROOT}/code')
import sh3_common as C; importlib.reload(C)
import numpy as np, pandas as pd
R = C.Run(ROOT, 'B6', 'Virtual saturation mutagenesis', resume=RESUME)
print(C.env_info())

Mounted at /content/drive
B6 — Virtual saturation mutagenesis
  results -> /content/drive/MyDrive/sh3_repro/results/B6_virtual_saturation_mutagenesis
{'python': '3.13.15', 'numpy': '2.1.3', 'pandas': '2.2.3', 'scipy': '1.16.3', 'sklearn': '1.6.1', 'transformers': '5.16.1', 'Bio': '1.88', 'torch': '2.11.0+cpu', 'gpu': 'none'}


In [2]:
from Bio import Align
from Bio.Align import substitution_matrices
EMB = C.load_embeddings(R.res('A2', 'esm_domain_embeddings.npz'))
LO = C.load_embeddings(R.res('A2', 'esm_logodds.npz'))
AT = pd.read_parquet(R.res('A2', 'features_atlas.parquet'))
STR = pd.read_csv(R.res('A2', 'structure_features.csv'))
DD = pd.read_parquet(R.res('A2', 'features_ddpca.parquet'), columns=['domain_key', 'wt_seq'])
FOLD, SC_M, SC_SD = C.load_fold_models(R.res('B1', 'models.pt'))
OBS = pd.read_parquet(R.res('B3', 'atlas_predictions.parquet'))
Q90 = json.load(open(R.res('B3', 'metrics_B3.json')))['metrics']['conformal_q90']; EPS = 0.05
DOMS = AT.drop_duplicates('domain_key')[['domain_key', 'gene', 'uniprot', 'domain_idx', 'wt_seq']].reset_index(drop=True)
print(f'{len(DOMS)} domains, {DOMS.wt_seq.str.len().sum():,} positions, {19 * DOMS.wt_seq.str.len().sum():,} substitutions')

310 domains, 19,272 positions, 366,168 substitutions


## 1. Every substitution, same features, same model

In [3]:
AAS = list('ACDEFGHIKLMNPQRSTVWY'); AAPOS = {a: i for i, a in enumerate(AAS)}
B62 = substitution_matrices.load('BLOSUM62')
rows = []
for k, s in zip(DOMS.domain_key, DOMS.wt_seq):
    for p, w in enumerate(s, 1):
        for m in AAS:
            if m != w: rows.append((k, p, w, m))
V = pd.DataFrame(rows, columns=['domain_key', 'pos', 'wt_aa', 'mut_aa']).merge(DOMS, on='domain_key')
lo = np.stack([LO[k][p - 1] for k, p in zip(V.domain_key, V.pos)]); r = np.arange(len(V))
w = V.wt_aa.map(AAPOS).values; mu = V.mut_aa.map(AAPOS).values
V['esm_score'] = lo[r, mu] - lo[r, w]; V['esm_wt_logp'] = lo[r, w]; V['esm_entropy'] = -(np.exp(lo) * lo).sum(1); del lo
V['dom_len'] = V.wt_seq.str.len(); V['rel_pos'] = V.pos / V.dom_len
V['d_hydropathy'] = V.mut_aa.map(C.KD) - V.wt_aa.map(C.KD); V['d_volume'] = V.mut_aa.map(C.VOL) - V.wt_aa.map(C.VOL)
V['d_charge'] = V.mut_aa.map(C.CHG).fillna(0) - V.wt_aa.map(C.CHG).fillna(0)
V['blosum62'] = [B62[a][b] for a, b in zip(V.wt_aa, V.mut_aa)]
V['to_proline'] = (V.mut_aa == 'P').astype(int); V['from_glycine'] = (V.wt_aa == 'G').astype(int); V['to_glycine'] = (V.mut_aa == 'G').astype(int)
X, V = C.build_X(V, EMB)
P = C.fold_scores(FOLD, X, SC_M, SC_SD, clip=10); del X
V['pred_ddG_fold'], V['sd_fold'] = P.mean(0), P.std(0)
V['fold_lo'] = V.pred_ddG_fold - Q90 * (V.sd_fold + EPS); V['destabilising'] = V.fold_lo >= 1.0
# consistency: observed atlas variants must get exactly the B1/B3 prediction
chk = OBS[['domain_key', 'pos', 'wt_aa', 'mut_aa', 'pred_ddG_fold']].merge(V[['domain_key', 'pos', 'wt_aa', 'mut_aa', 'pred_ddG_fold']],
                                                                           on=['domain_key', 'pos', 'wt_aa', 'mut_aa'], suffixes=('_obs', '_virtual'))
MAXDIFF = float((chk.pred_ddG_fold_obs - chk.pred_ddG_fold_virtual).abs().max())
print(f'{len(V):,} virtual variants | {len(chk):,} observed variants matched, max prediction difference {MAXDIFF:.2e}')
assert len(chk) == len(OBS) and MAXDIFF < 1e-4, 'virtual scan does not reproduce the observed-variant predictions'

366,168 virtual variants | 6,884 observed variants matched, max prediction difference 9.54e-07


## 2. Which substitutions can a single nucleotide change reach?

In [4]:
CODE = {'TTT':'F','TTC':'F','TTA':'L','TTG':'L','CTT':'L','CTC':'L','CTA':'L','CTG':'L','ATT':'I','ATC':'I','ATA':'I','ATG':'M',
        'GTT':'V','GTC':'V','GTA':'V','GTG':'V','TCT':'S','TCC':'S','TCA':'S','TCG':'S','CCT':'P','CCC':'P','CCA':'P','CCG':'P',
        'ACT':'T','ACC':'T','ACA':'T','ACG':'T','GCT':'A','GCC':'A','GCA':'A','GCG':'A','TAT':'Y','TAC':'Y','CAT':'H','CAC':'H',
        'CAA':'Q','CAG':'Q','AAT':'N','AAC':'N','AAA':'K','AAG':'K','GAT':'D','GAC':'D','GAA':'E','GAG':'E','TGT':'C','TGC':'C',
        'TGG':'W','CGT':'R','CGC':'R','CGA':'R','CGG':'R','AGT':'S','AGC':'S','AGA':'R','AGG':'R','GGT':'G','GGC':'G','GGA':'G','GGG':'G'}
reach = set()
for cod, a in CODE.items():
    for i in range(3):
        for b in 'ACGT':
            c2 = cod[:i] + b + cod[i + 1:]
            if c2 in CODE and CODE[c2] != a: reach.add((a, CODE[c2]))
V['snv_reachable'] = [(a, b) in reach for a, b in zip(V.wt_aa, V.mut_aa)]
obs_keys = set(zip(OBS.domain_key, OBS.pos, OBS.mut_aa)); V['observed_in_cancer'] = [(k, p, m) in obs_keys for k, p, m in zip(V.domain_key, V.pos, V.mut_aa)]
print(f"single-nucleotide-reachable: {V.snv_reachable.mean():.1%} of substitutions; observed cancer variants reachable: "
      f"{V[V.observed_in_cancer].snv_reachable.mean():.1%}")

single-nucleotide-reachable: 41.0% of substitutions; observed cancer variants reachable: 99.7%


## 3. The family landscape and where cancer mutations sit

In [5]:
al = Align.PairwiseAligner(mode='global', open_gap_score=-11, extend_gap_score=-1); al.match_score = 2; al.mismatch_score = -1
ref_seq = DD[DD.domain_key == 'GRB2-SH3'].wt_seq.iat[0]
def to_ref(seq):
    a = al.align(ref_seq, seq)[0]
    return {qs + k + 1: rs_ + k + 1 for (rs_, re_), (qs, qe) in zip(a.aligned[0], a.aligned[1]) for k in range(re_ - rs_)}
REFMAP = {k: to_ref(s) for k, s in zip(DOMS.domain_key, DOMS.wt_seq)}
V['ref_pos'] = [REFMAP[k].get(int(p), np.nan) for k, p in zip(V.domain_key, V.pos)]
ref = STR[STR.domain_key == 'GRB2-SH3'].set_index('pos')
FAM = V.dropna(subset=['ref_pos']).groupby('ref_pos').agg(domains=('domain_key', 'nunique'), mean_pred=('pred_ddG_fold', 'mean'),
                                                        frac_destabilising=('destabilising', 'mean'),
                                                        observed_variants=('observed_in_cancer', 'sum')).reset_index()
FAM['rsasa'] = FAM.ref_pos.map(ref.rsasa); FAM['buried'] = FAM.rsasa < 0.25; FAM['interface'] = FAM.ref_pos.map(ref.dist_peptide) < 5.0
FAM.round(4).to_csv(R.out('family_position_map.csv'), index=False)
DSUM = V.groupby(['domain_key', 'gene']).agg(length=('pos', 'max'), frac_destabilising=('destabilising', 'mean'),
                                            mean_pred=('pred_ddG_fold', 'mean'), observed=('observed_in_cancer', 'sum')).reset_index()
DSUM.to_csv(R.out('domain_summary.csv'), index=False)
cmp_rows = []
for label, pool in [('all possible substitutions', V), ('single-nucleotide-reachable only', V[V.snv_reachable])]:
    cmp_rows.append(dict(set=label, variants=len(pool), frac_destabilising=pool.destabilising.mean(),
                         median_pred=pool.pred_ddG_fold.median()))
o = V[V.observed_in_cancer]
cmp_rows.append(dict(set='observed in cancer', variants=len(o), frac_destabilising=o.destabilising.mean(), median_pred=o.pred_ddG_fold.median()))
CMPT = pd.DataFrame(cmp_rows)
# within the same domains and the reachable set: are observed variants depleted of damage? (permutation, matched on domain)
pool = V[V.snv_reachable]; rng = np.random.default_rng(0); need = o[o.snv_reachable].domain_key.value_counts()
idx = {k: np.flatnonzero(pool.domain_key.values == k) for k in need.index}; pv = pool.destabilising.values
null = np.array([sum(pv[rng.choice(idx[k], n, replace=False)].sum() for k, n in need.items()) / need.sum() for _ in range(1000)])
OBS_FR = float(o[o.snv_reachable].destabilising.mean()); P_LOW = float((np.sum(null <= OBS_FR) + 1) / 1001)
V.drop(columns=['wt_seq']).to_parquet(R.out('sh3_virtual_dms.parquet'), index=False)
print(CMPT.round(4).to_string(index=False)); print(f'observed {OBS_FR:.4f} vs domain-matched reachable null median {np.median(null):.4f}, p(depleted) = {P_LOW:.3g}')

                             set  variants  frac_destabilising  median_pred
      all possible substitutions    366168              0.1210       0.7716
single-nucleotide-reachable only    150040              0.0923       0.6543
              observed in cancer      6884              0.0596       0.5459
observed 0.0593 vs domain-matched reachable null median 0.0918, p(depleted) = 0.000999


## Results

In [6]:
for k, v in [('virtual_variants', len(V)), ('domains', V.domain_key.nunique()), ('max_diff_vs_observed', MAXDIFF),
             ('frac_destabilising_all_possible', CMPT.frac_destabilising[0]), ('frac_destabilising_reachable', CMPT.frac_destabilising[1]),
             ('frac_destabilising_observed', CMPT.frac_destabilising[2]), ('frac_destabilising_observed_reachable', OBS_FR),
             ('matched_null_median', float(np.median(null))), ('p_observed_depleted', P_LOW),
             ('rho_family_damage_vs_burial', C.spearman(FAM.dropna(subset=['rsasa']).mean_pred, -FAM.dropna(subset=['rsasa']).rsasa))]:
    R.metric(k, v)
R.section('Observed cancer variants against every possible substitution', 'Destabilising = lower 90% interval bound ≥ 1 kcal/mol.', CMPT, '{:.4f}')
R.section('Domain-matched test', f'Observed single-nucleotide-reachable variants: {OBS_FR:.2%} destabilising; random reachable '
          f'substitutions from the same domains: median {np.median(null):.2%} (1,000 draws); one-sided p that observed variants are '
          f'depleted of damage = {P_LOW:.3g}. Depletion would be the signature of purifying selection; enrichment of positive '
          'selection. Codon usage and trinucleotide context are not modelled.')
R.section('Family map: the 15 most damage-sensitive aligned positions', '', FAM.sort_values('mean_pred', ascending=False).head(15), '{:.3f}')
R.finish([f"{len(V):,} virtual substitutions predicted across {V.domain_key.nunique()} SH3 domains; the observed variants reproduce B1 "
          f"exactly (max difference {MAXDIFF:.1e}).",
          f"{CMPT.frac_destabilising[0]:.1%} of all possible substitutions are called destabilising "
          f"({CMPT.frac_destabilising[1]:.1%} of single-nucleotide-reachable ones) against {CMPT.frac_destabilising[2]:.1%} of observed cancer variants.",
          f"Domain-matched: observed {OBS_FR:.2%} vs expected {np.median(null):.2%} (p depleted = {P_LOW:.2g}).",
          'A citable resource: a predicted folding landscape for every human SH3 domain in the atlas, with intervals.'])


B6: no recorded values to compare
Empty DataFrame
Columns: [quantity, this_run, recorded, status]
Index: []

wrote RESULTS_B6.md, metrics_B6.json, comparison_B6.csv


,quantity,this_run,recorded,status
0,virtual_variants,3.661680e+05,None,
1,domains,3.100000e+02,None,
2,max_diff_vs_observed,9.536743e-07,None,
3,frac_destabilising_all_possible,1.209854e-01,None,
4,frac_destabilising_reachable,9.231538e-02,None,
5,frac_destabilising_observed,5.955840e-02,None,
6,frac_destabilising_observed_reachable,5.928623e-02,None,
7,matched_null_median,9.176985e-02,None,
8,p_observed_depleted,9.990010e-04,None,
9,rho_family_damage_vs_burial,7.753142e-01,None,
